# Heat Carbon — Analise e Visualizacao
**Projeto:** Projetos 2 — Cesar School  
**Empresa parceira:** Edenred Brasil (Taggy)

In [ ]:
import os, sys, json, warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium

warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath('')
sys.path.insert(0, BASE_DIR)

from banco import carregar_dados, DB_PATH
sns.set_theme(style='whitegrid')
print('Banco:', DB_PATH)

## 1. Carregando os dados

In [ ]:
df = carregar_dados()
print(f'Total de registros: {len(df)}')
df.head()

## 2. Metricas gerais

In [ ]:
if df.empty:
    print('Banco vazio — cadastre usuarios com cadastro.py e rode novamente.')
else:
    total_co2         = df['co2e_total_kg'].sum()
    total_reg         = len(df)
    bairro_lider      = df.groupby('bairro')['co2e_total_kg'].sum().idxmax()
    media_por_usuario = df['co2e_total_kg'].mean()
    print(f'Total CO2e evitado:   {total_co2:,.2f} kg')
    print(f'Total de registros:   {total_reg}')
    print(f'Media por usuario:    {media_por_usuario:.2f} kg')
    print(f'Bairro lider:         {bairro_lider}')

## 3. CO2e por bairro — ranking

In [ ]:
if not df.empty:
    por_bairro = (
        df.groupby('bairro')['co2e_total_kg']
        .sum().reset_index()
        .sort_values('co2e_total_kg', ascending=False)
        .reset_index(drop=True)
    )
    por_bairro.index += 1
    por_bairro.columns = ['Bairro', 'CO2e evitado (kg)']
    display(por_bairro.head(10))

## 4. Mapa — impacto por bairro

In [ ]:
with open('data/bairros_recife.geojson', encoding='utf-8') as f:
    geo = json.load(f)

mapa = folium.Map(location=[-8.054, -34.881], zoom_start=12, tiles='CartoDB positron')

if not df.empty:
    df_mapa = df.groupby('bairro')['co2e_total_kg'].sum().reset_index()
    folium.Choropleth(
        geo_data=geo, data=df_mapa,
        columns=['bairro', 'co2e_total_kg'],
        key_on='feature.properties.EBAIRRNOMEOF',
        fill_color='YlGn', fill_opacity=0.75, line_opacity=0.4,
        legend_name='CO2e evitado (kg)', nan_fill_color='lightgray',
    ).add_to(mapa)
    co2_dict = df_mapa.set_index('bairro')['co2e_total_kg'].to_dict()
    for feat in geo['features']:
        feat['properties']['co2e'] = round(co2_dict.get(feat['properties']['EBAIRRNOMEOF'], 0), 1)
    folium.GeoJson(
        geo,
        style_function=lambda x: {'fillOpacity': 0, 'color': 'transparent', 'weight': 0},
        tooltip=folium.GeoJsonTooltip(fields=['EBAIRRNOMEOF', 'co2e'], aliases=['Bairro', 'CO2e evitado (kg)'])
    ).add_to(mapa)

mapa

## 5. Top 15 bairros

In [ ]:
if not df.empty:
    top15 = df.groupby('bairro')['co2e_total_kg'].sum().nlargest(15).reset_index()
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=top15, y='bairro', x='co2e_total_kg', hue='bairro', palette='YlGn_r', legend=False, ax=ax)
    ax.set_title('Heat Carbon - Top 15 bairros por CO2e evitado', fontsize=13)
    ax.set_xlabel('CO2e evitado (kg)')
    ax.set_ylabel('Bairro')
    plt.tight_layout()
    plt.show()

## 6. CO2e por tipo de veiculo

In [ ]:
if not df.empty:
    por_veiculo = df.groupby('tipo_veiculo')['co2e_total_kg'].sum().reset_index().sort_values('co2e_total_kg', ascending=False)
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=por_veiculo, x='tipo_veiculo', y='co2e_total_kg', hue='tipo_veiculo', palette='Blues_d', legend=False, ax=ax)
    ax.set_title('Heat Carbon - CO2e evitado por tipo de veiculo', fontsize=13)
    ax.set_xlabel('Tipo de veiculo')
    ax.set_ylabel('CO2e evitado (kg)')
    plt.tight_layout()
    plt.show()

## 7. CO2e por combustivel

In [ ]:
if not df.empty:
    por_combustivel = df.groupby('combustivel')['co2e_total_kg'].sum().reset_index().sort_values('co2e_total_kg', ascending=False)
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=por_combustivel, x='combustivel', y='co2e_total_kg', hue='combustivel', palette='Greens_d', legend=False, ax=ax)
    ax.set_title('Heat Carbon - CO2e evitado por combustivel', fontsize=13)
    ax.set_xlabel('Combustivel')
    ax.set_ylabel('CO2e evitado (kg)')
    plt.tight_layout()
    plt.show()

## 8. Evolucao no tempo

In [ ]:
if not df.empty:
    df['data'] = pd.to_datetime(df['registrado_em']).dt.date
    por_dia = df.groupby('data')['co2e_total_kg'].sum().reset_index()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(por_dia['data'], por_dia['co2e_total_kg'], marker='o', color='green')
    ax.fill_between(por_dia['data'], por_dia['co2e_total_kg'], alpha=0.2, color='green')
    ax.set_title('Heat Carbon - CO2e evitado por dia', fontsize=13)
    ax.set_xlabel('Data')
    ax.set_ylabel('CO2e evitado (kg)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()